In [ ]:
!pip install flask transformers requests

In [ ]:
%%writefile flask_chatbot_app.py
from flask import Flask, request, jsonify
from transformers import pipeline

app = Flask(__name__)

# Load the Hugging Face model

chatbot_model = pipeline("text-generation", model="microsoft/Phi-3-mini-4k-instruct", trust_remote_code=True)
conversation_history = []

@app.route('/chatbot', methods=['POST'])
def chatbot():
    user_input = request.json.get('message')
    if not user_input:
        return jsonify({'response': 'Please provide a message.'}), 400
    response = generate_response(user_input)
    return jsonify({'response': response})

def generate_response(user_input):
    # Generate a response using the Hugging Face model
    conversation_history.append(
        {"role": "user", "content": user_input},
    )
    result = chatbot_model(conversation_history, num_return_sequences=1, max_new_tokens=250)
    conversation_history.append(
        {"role": "assistant", "content": result[0]['generated_text'][-1]['content']},
    )
    return result[0]['generated_text'][-1]['content']

if __name__ == '__main__':
    app.run(debug=True, port=5000, host='0.0.0.0')


In [ ]:

import subprocess

# Stop any running Flask app
subprocess.run(['pkill', '-f', 'flask_chatbot_app.py'])

In [ ]:
!nohup python flask_chatbot_app.py &

In [ ]:
!sudo lsof -i -P -n | grep LISTEN

In [ ]:

import requests

# Define the URL of the Flask app
url = 'http://127.0.0.1:5000/chatbot'

# Send a request to the Flask app
response = requests.post(url, json={'message': 'Hello, how are you?'})
print(response.json())


In [ ]:
response = requests.post(url, json={'message': 'I was wondering how can I go to Eiffel Tower from the airport using the train and subway?'})
print(response.json())

In [ ]:
response = requests.post(url, json={'message': 'And how do I go to Arche du Triomphe from there?'})
print(response.json())